# 面向地理空间的Segment Anything

## 介绍

传统的遥感图像分割需要收集标注的训练数据，训练模型，并希望它能推广到新场景。每个新的地理区域、传感器或季节都可能意味着重新开始标注过程。

Meta AI于2023年4月发布的[Segment Anything Model](https://ai.meta.com/datasets/segment-anything)（SAM）通过实现零样本分割改变了这一局面——无需特定任务训练即可分割几乎任何图像中的任何对象。最新一代的SAM 3在此基础上增强了分割质量，支持多模态提示，并采用流内存架构来跨视频帧跟踪对象。

单个模型可以描绘内罗毕的建筑轮廓、绘制爱荷华州的农田地图，并追踪亚马逊的河流网络，而无需看到这些地区的任何标注示例。虽然SAM 3不能取代领域专业知识，但它极大地加速了许多地理空间工作流中最耗时的阶段。

segment-geospatial Python包（`samgeo`）通过`SamGeo3`和`SamGeo3Video`类连接SAM 3和地理空间领域。这些类处理地理参考输入、坐标系和矢量输出格式，使预测结果可以直接集成到GIS工作流中。

本教程演示`samgeo`为SAM 3提供的实际工作流，包括文本、点和框提示；切片和批量分割；交互式分割；以及视频对象跟踪。

## 学习目标

在本教程结束时，您将能够：

- 解释SAM 3如何进行零样本图像和视频分割
- 使用`SamGeo3`通过文本、点和框提示分割卫星图像
- 将带有置信度分数的分割掩码保存为地理参考的GeoTIFF文件
- 使用共享工作流批量处理多个图像
- 使用切片方法分割大型GeoTIFF图像
- 使用内置地图界面交互式分割图像
- 使用`SamGeo3Video`跨视频帧跟踪对象

## SAM 3工作原理

SAM 3的架构由三个核心图像分割组件组成，加上一个用于视频的流内存模块。

**图像编码器。** 视觉Transformer（ViT）将输入图像处理成高维特征嵌入。这个耗时的步骤每张图像只运行一次，并且能很好地推广到未见过的图像，包括卫星和航空照片。

**提示编码器。** 提示编码器将用户提供的提示（文本描述、点坐标或边界框）转换为模型可以与图像嵌入一起使用的格式。它很轻量，因此可以针对同一个缓存的嵌入快速测试不同的提示。

**掩码解码器。** 掩码解码器使用带有交叉注意力的修改版Transformer解码器，结合图像嵌入和编码后的提示来生成带置信度分数的最终分割掩码。

**视频流内存。** SAM 3维护一个包含先前看到的帧及其分割结果的内存库。当新帧到达时，内存库为分割提供信息，以便对象可以在数百帧中连贯地跟踪。

这种设计意味着昂贵的图像编码只进行一次，然后可以针对缓存的嵌入廉价地评估任意数量的提示。

## 设置环境

安装所需的包并导入本教程中使用的库。

In [ ]:
# %pip install "geoai-py[extra]" "segment-geospatial[samgeo3]"

In [ ]:
import os
import geoai
import leafmap
from samgeo import SamGeo3, SamGeo3Video, download_file, show_image
from samgeo.common import raster_to_vector, regularize

SAM 3在首次使用前需要通过Hugging Face获得访问权限。在[SAM 3模型页面](https://huggingface.co/facebook/sam3)申请访问。获得访问权限后，使用Hugging Face登录进行身份验证：

In [ ]:
# from huggingface_hub import login
# login()

## 图像分割

核心工作流遵循三个步骤：加载图像、使用提示生成掩码、保存结果。

下载覆盖加州大学伯克利分校的卫星图像样本：

In [ ]:
url = "https://data.source.coop/opengeos/geoai/uc-berkeley.tif"
image_path = download_file(url)

在交互式地图上显示卫星图像：

In [ ]:
m = leafmap.Map()
m.add_raster(image_path, layer_name="Satellite image")
m

初始化`SamGeo3`并加载图像。`set_image()`方法运行图像编码器一次并缓存嵌入：

In [ ]:
sam3 = SamGeo3(backend="meta", device=None, checkpoint_path=None, load_from_HF=True)
sam3.set_image(image_path)

### 文本提示分割

使用自然语言文本提示生成掩码：

In [ ]:
sam3.generate_masks(prompt="building")

将检测到的对象作为彩色叠加层可视化在原始图像上：

In [ ]:
sam3.show_anns()

使用`show_masks()`在空白背景上仅显示分割掩码：

In [ ]:
sam3.show_masks()

将掩码保存为GeoTIFF。设置`unique=True`为每个分割对象分配一个不同的整数值：

In [ ]:
sam3.save_masks(output="building_masks.tif", unique=True)

使用`save_scores`参数也捕获预测置信度：

In [ ]:
sam3.save_masks(
    output="building_masks_with_scores.tif",
    save_scores="building_scores.tif",
    unique=True,
)

按置信度分数着色显示掩码：

In [ ]:
sam3.show_masks(cmap="coolwarm")

在交互式地图上可视化置信度分数：

In [ ]:
m.add_raster("building_masks.tif", layer_name="Building masks", visible=False)
m.add_raster(
    "building_scores.tif",
    layer_name="Building scores",
    cmap="coolwarm",
    opacity=0.8,
    nodata=0,
    vmin=0.5,
    vmax=1.0,
)
m.add_colormap(cmap="coolwarm", vmin=0.5, vmax=1.0, label="Confidence score")
m

### 框提示分割

边界框提示告诉SAM 3使用框内的对象作为参考，并在图像的其他地方搜索相似对象。使用地理坐标以`[xmin, ymin, xmax, ymax]`格式指定框：

In [ ]:
# Define boxes in [xmin, ymin, xmax, ymax] format
boxes = [[-122.2597, 37.8709, -122.2587, 37.8717]]
box_labels = [True]  # True=include, False=exclude

sam3.generate_masks_by_boxes(boxes, box_labels, box_crs="EPSG:4326")

在图像上叠加边界框：

In [ ]:
sam3.show_boxes(boxes, box_labels, box_crs="EPSG:4326")

可视化分割掩码：

In [ ]:
sam3.show_anns()

将框提示分割掩码保存为地理参考的GeoTIFF：

In [ ]:
building_mask_path = "building_masks.tif"
sam3.save_masks(output=building_mask_path, unique=True)

在原始卫星图像上叠加掩码：

In [ ]:
geoai.view_raster(building_mask_path, nodata=0, opacity=0.7, basemap=image_path)

进入下一节前释放GPU内存：

In [ ]:
geoai.empty_cache()

## 实例分割的点提示

点提示使用前景点（标签=1）和背景点（标签=0）分割特定对象。使用`enable_inst_interactivity=True`初始化`SamGeo3`以启用此模式。

In [ ]:
url = "https://data.source.coop/opengeos/geoai/truck-example.jpg"
image_path = download_file(url)

显示带有轴标签的图像以便读取像素坐标：

In [ ]:
show_image(image_path, axis="on")

初始化带有实例交互功能的SAM 3并加载图像：

In [ ]:
sam = SamGeo3(backend="meta", enable_inst_interactivity=True)
sam.set_image(image_path)

### 单点提示

以`[x, y]`像素坐标指定单个前景点：

In [ ]:
sam.generate_masks_by_points([[750, 370]])

在图像上叠加点标记：

In [ ]:
sam.show_points([[750, 370]], [1])

可视化生成的分割掩码：

In [ ]:
sam.show_anns()

将掩码保存到PNG文件：

In [ ]:
sam.save_masks("truck_mask.png", unique=True)

### 多点提示

同一对象上的多个前景点改善大型或不规则形状对象的覆盖：

In [ ]:
sam.generate_masks_by_points([[500, 375], [1125, 625]], point_labels=[1, 1])

叠加两个点标记：

In [ ]:
sam.show_points([[500, 375], [1125, 625]], [1, 1])

可视化生成的分割掩码：

In [ ]:
sam.show_anns()

### 背景点

背景点（标签=0）从掩码中排除一个区域以细化模糊边界：

In [ ]:
sam.generate_masks_by_points([[750, 370], [1125, 625]], point_labels=[1, 0])

叠加前景（绿色）和背景（红色）点标记：

In [ ]:
sam.show_points([[750, 370], [1125, 625]], [1, 0])

可视化细化后的分割掩码：

In [ ]:
sam.show_anns()

### 多框提示

可以同时处理多个框以在一次调用中分割多个对象：

In [ ]:
boxes = [
    [75, 275, 1725, 850],  # Whole truck
    [425, 600, 700, 875],  # Rear wheel
    [1375, 550, 1650, 800],  # Front wheel on the passenger side
    [1240, 675, 1400, 750],  # Front wheel on the driver's side
]
sam.generate_masks_by_boxes_inst(boxes)

在图像上叠加边界框：

In [ ]:
sam.show_boxes(boxes)

可视化分割掩码：

In [ ]:
sam.show_anns()

保存多框分割掩码：

In [ ]:
sam.save_masks("truck_boxes_mask.png", unique=True)

继续前释放GPU内存：

In [ ]:
geoai.empty_cache()

### 地理空间数据的批量点提示

对于地理参考图像，提供地理坐标作为点提示。`generate_masks_by_points_patch()`方法在每个点分割对象并保存地理参考结果。

下载卫星图像和建筑质心GeoJSON：

In [ ]:
image_url = "https://data.source.coop/opengeos/geoai/wa-building-image.tif"
geojson_url = "https://data.source.coop/opengeos/geoai/wa-building-centroids.geojson"
image_path = download_file(image_url)
geojson_path = download_file(geojson_url)

显示卫星图像：

In [ ]:
m = leafmap.Map()
m.add_raster(image_path, layer_name="Satellite image")
m

初始化SAM 3并设置图像：

In [ ]:
sam = SamGeo3(backend="meta", enable_inst_interactivity=True)
sam.set_image(image_path)

在每个点位置分割建筑：

In [ ]:
point_coords_batch = [
    [-117.599896, 47.655345],
    [-117.59992, 47.655167],
    [-117.599928, 47.654974],
    [-117.599518, 47.655337],
]

sam.generate_masks_by_points_patch(
    point_coords_batch=point_coords_batch,
    point_crs="EPSG:4326",
    output="masks.tif",
    dtype="uint8",
)

在图像上叠加点标记：

In [ ]:
sam.show_points(point_coords_batch, point_crs="EPSG:4326")

将掩码添加到交互式地图：

In [ ]:
m.add_raster("masks.tif", cmap="viridis", nodata=0, opacity=0.7, layer_name="Mask")
m

您也可以直接提供包含点几何的GeoJSON文件：

In [ ]:
sam.generate_masks_by_points_patch(
    point_coords_batch=geojson_path,
    point_crs="EPSG:4326",
    output="building_masks.tif",
    dtype="uint16",
)

将建筑掩码和质心标记添加到地图：

In [ ]:
m.add_raster(
    "building_masks.tif", cmap="jet", nodata=0, opacity=0.7, layer_name="Building masks"
)
m.add_circle_markers_from_xy(
    geojson_path, radius=3, color="red", fill_color="yellow", fill_opacity=0.8
)
m

继续前释放GPU内存：

In [ ]:
geoai.empty_cache()

## 建筑提取的框提示

框提示适用于建筑轮廓提取，因为建筑物具有明确的矩形范围。本节演示从分割到矢量导出和正则化的完整工作流。

显示卫星图像：

In [ ]:
m = leafmap.Map()
m.add_raster(image_path, layer_name="Satellite image")
m

初始化SAM 3：

In [ ]:
sam = SamGeo3(backend="meta", enable_inst_interactivity=True)
sam.set_image(image_path)

在地理坐标中定义建筑物周围的边界框：

In [ ]:
if m.user_rois is not None:
    boxes = m.user_rois
else:
    boxes = [
        [-117.5995, 47.6518, -117.5988, 47.652],
        [-117.5987, 47.6518, -117.5979, 47.652],
    ]

使用边界框生成掩码：

In [ ]:
sam.generate_masks_by_boxes_inst(boxes=boxes, box_crs="EPSG:4326")

将建筑掩码保存为GeoTIFF：

In [ ]:
sam.save_masks(output="mask.tif", dtype="uint8")

在地图上叠加掩码：

In [ ]:
m.add_raster("mask.tif", cmap="viridis", nodata=0, opacity=0.5, layer_name="Mask")
m

### 使用矢量文件作为框提示

无需指定单个坐标，直接传递GeoJSON或Shapefile：

In [ ]:
url = "https://data.source.coop/opengeos/geoai/wa-building-bboxes.geojson"
geojson_path = download_file(url)

在卫星图像上叠加边界框：

In [ ]:
m = leafmap.Map()
m.add_raster(image_path, layer_name="Image")
style = {
    "color": "#ffff00",
    "weight": 2,
    "fillColor": "#7c4185",
    "fillOpacity": 0,
}
m.add_vector(geojson_path, style=style, zoom_to_layer=True, layer_name="Bboxes")
m

使用GeoJSON文件作为框提示为所有建筑物生成掩码：

In [ ]:
output_masks = "building_masks.tif"
sam.generate_masks_by_boxes_inst(
    boxes=geojson_path,
    box_crs="EPSG:4326",
    output=output_masks,
    dtype="uint16",
    multimask_output=False,
)

显示建筑掩码：

In [ ]:
m.add_raster(
    output_masks, cmap="jet", nodata=0, opacity=0.5, layer_name="Building masks"
)
m

### 转换为矢量并正则化

将栅格掩码转换为矢量多边形：

In [ ]:
output_vector = "building_vector.geojson"
raster_to_vector(output_masks, output_vector)

`regularize()`函数调整多边形几何以产生更干净、更规则的轮廓：

In [ ]:
output_regularized = "building_regularized.geojson"
regularize(output_vector, output_regularized)

将正则化的轮廓添加到地图：

In [ ]:
m.add_vector(
    output_regularized, style=style, layer_name="Building regularized", info_mode=None
)
m

继续前释放GPU内存：

In [ ]:
geoai.empty_cache()

## 批量分割

批量分割在单个工作流中使用相同提示处理多个图像。

下载覆盖加州大学伯克利分校校园的四个卫星图像瓦片：

In [ ]:
image_paths = []
for i in range(1, 5):
    url = f"https://data.source.coop/opengeos/geoai/uc-berkeley-{i}.tif"
    image_path = download_file(url)
    image_paths.append(image_path)

在交互式地图上显示所有四个瓦片：

In [ ]:
m = leafmap.Map()
for i, image_path in enumerate(image_paths):
    m.add_raster(image_path, layer_name=f"image_{i + 1}")
m

初始化SAM 3并批量加载所有图像：

In [ ]:
sam3 = SamGeo3(backend="meta", device=None, checkpoint_path=None, load_from_HF=True)
sam3.set_image_batch(image_paths)

使用单个文本提示为所有图像生成掩码：

In [ ]:
sam3.generate_masks_batch("building", min_size=100)

```text
处理了4张图像，共找到174个对象。
```

检查每张图像中检测到的对象数量：

In [ ]:
for i, result in enumerate(sam3.batch_results):
    print(f"Image {i + 1}: Found {len(result['masks'])} objects")

```text
图像1：找到46个对象
图像2：找到50个对象
图像3：找到64个对象
图像4：找到14个对象
```

在网格布局中可视化所有结果：

In [ ]:
sam3.show_anns_batch(ncols=2, show_bbox=True, show_score=True, figsize=(12, 8))

将标注图像保存到磁盘：

In [ ]:
sam3.show_anns_batch(output_dir="output/annotations/", prefix="ann", dpi=300)

将每张图像的掩码导出为单独的地理参考GeoTIFF：

In [ ]:
saved_files = sam3.save_masks_batch(
    output_dir="output/", prefix="building_mask", unique=True
)

继续前释放GPU内存：

In [ ]:
geoai.empty_cache()

## 大图像的切片分割

大型卫星图像通常超过GPU内存限制。`generate_masks_tiled()`方法将图像分割成重叠的切片，独立处理每个切片，并合并结果。

下载用于水体检测的大型NAIP图像：

In [ ]:
url = "https://data.source.coop/opengeos/geoai/naip_water_train.tif"
image_path = download_file(url)

检查图像尺寸：

In [ ]:
geoai.print_raster_info(image_path)

初始化SAM 3并运行切片分割：

In [ ]:
sam = SamGeo3(backend="meta")

output_path = "segmentation_mask.tif"

sam.generate_masks_tiled(
    source=image_path,
    prompt="water",
    output=output_path,
    tile_size=1024,
    overlap=128,
    min_size=100,
    unique=False,
    dtype="int32",
    verbose=True,
)

可视化水体分割结果：

In [ ]:
m = leafmap.Map()
m.add_raster(image_path, layer_name="Original Image")
m.add_raster(
    output_path, nodata=0, opacity=0.8, cmap="Blues", layer_name="Segmentation Mask"
)
m

将栅格掩码转换为矢量多边形：

In [ ]:
vector_path = "segmentation_mask.gpkg"
geoai.raster_to_vector(output_path, vector_path)

平滑多边形边界以减少像素化伪影：

In [ ]:
smooth_vector_path = "segmentation_mask_smooth.gpkg"
gdf = geoai.smooth_vector(vector_path, smooth_vector_path)

将平滑后的多边形添加到地图：

In [ ]:
style = {
    "color": "#ff0000",
    "weight": 2,
    "fillOpacity": 0,
}
m.add_gdf(gdf, layer_name="Smoothed Vector", info_mode=None, style=style)
m

继续前释放GPU内存：

In [ ]:
geoai.empty_cache()

调整切片分割时，考虑这些参数：

- **tile_size**：较大的切片捕获更多上下文但需要更多GPU内存。从1024开始。
- **overlap**：较高的重叠（128-256像素）防止边界伪影，但会增加处理时间。
- **min_size / max_size**：使用这些参数过滤噪声或不相关的大区域。
- **dtype**：对于许多对象使用`int32`，最多65,535个对象使用`uint16`，或对于二值掩码使用`uint8`。

## 交互式分割

`show_map()`方法启动一个交互式Jupyter小部件，结合了地图界面和SAM 3分割功能。

In [ ]:
url = "https://data.source.coop/opengeos/geoai/uc-berkeley.tif"
image_path = download_file(url)

使用Hugging Face Transformers后端初始化SAM 3：

In [ ]:
sam3 = SamGeo3(
    backend="transformers", device=None, checkpoint_path=None, load_from_HF=True
)
sam3.set_image(image_path)

生成初始掩码集：

In [ ]:
sam3.generate_masks(prompt="building")
sam3.save_masks("masks.tif")

启动交互式界面。输入文本提示或绘制矩形，然后点击**Segment**：

In [ ]:
sam3.show_map(height="700px", min_size=10)

界面支持文本提示模式和边界框模式。无需重新运行图像编码器即可更新结果。

继续前释放GPU内存：

In [ ]:
geoai.empty_cache()

## 视频分割

`SamGeo3Video`将SAM 3扩展到视频，支持MP4文件、JPEG帧目录和GeoTIFF文件目录。

### 文本提示视频分割

下载汽车样本视频：

In [ ]:
url = "https://data.source.coop/opengeos/geoai/cars.mp4"
video_path = download_file(url)

初始化`SamGeo3Video`，加载视频并预览：

In [ ]:
sam = SamGeo3Video()
sam.set_video(video_path)
sam.show_video(video_path)

使用文本提示生成掩码：

In [ ]:
sam.generate_masks("car")

显示第一帧以验证检测：

In [ ]:
sam.show_frame(0, axis="on")

显示每20帧采样的帧网格：

In [ ]:
sam.show_frames(frame_stride=20, ncols=3)

按ID移除不需要的对象并重新传播：

In [ ]:
sam.remove_object(2)
sam.propagate()
sam.show_frame(0)

保存每帧掩码：

In [ ]:
os.makedirs("output", exist_ok=True)
sam.save_masks("output/masks")

渲染标注视频：

In [ ]:
sam.save_video("output/segmented.mp4", fps=25)

关闭视频会话：

In [ ]:
sam.close()

### 点提示视频分割

点提示提供对要跟踪的对象的精确控制。

初始化新的视频会话：

In [ ]:
sam = SamGeo3Video()
sam.set_video(video_path)
sam.init_tracker()
sam.show_frame(0, axis="on")

为每个对象添加点提示并传播：

In [ ]:
sam.add_point_prompts([[300, 200]], [1], obj_id=1, frame_idx=0)
sam.add_point_prompts([[420, 200]], [1], obj_id=2, frame_idx=0)
sam.propagate()

显示带有跟踪掩码的第一帧：

In [ ]:
sam.show_frame(0, axis="on")

在采样帧上可视化跟踪：

In [ ]:
sam.show_frames(frame_stride=20, ncols=3)

结合使用正点和负点来细化掩码：

In [ ]:
# Positive point on windshield, negative point on car body
sam.add_point_prompts(
    points=[[335, 195], [335, 220]],
    labels=[1, 0],
    obj_id=1,
    frame_idx=0,
)
sam.propagate()
sam.show_frames(frame_stride=20, ncols=3)

保存细化结果并关闭会话：

In [ ]:
sam.save_masks("output/masks")
sam.save_video("output/segmented.mp4", fps=25)
sam.close()

### 对象跟踪

SAM 3跨视频帧跟踪多个对象，适用于监控车辆、运动员和其他移动目标。

下载篮球视频：

In [ ]:
url = "https://data.source.coop/opengeos/geoai/basketball.mp4"
video_path = download_file(url)

初始化并预览视频：

In [ ]:
sam = SamGeo3Video()
sam.set_video(video_path)
sam.show_video(video_path)

使用文本提示检测并跟踪所有球员：

In [ ]:
sam.generate_masks("player")

创建显示名称标签并可视化第一帧：

In [ ]:
player_names = {}
for i in range(15):
    player_names[i] = f"Player {i}"
sam.show_frame(0, axis="on", show_ids=player_names)

移除虚假检测：

In [ ]:
sam.remove_object(obj_id=[5, 8, 12, 13])
sam.propagate()
sam.show_frame(0, show_ids=player_names)

保存掩码，渲染标注视频并预览：

In [ ]:
os.makedirs("output", exist_ok=True)
sam.save_masks("output/masks")
sam.save_video("output/players_segmented.mp4", fps=60, show_ids=player_names)
sam.show_video("output/players_segmented.mp4")

关闭会话并释放GPU内存：

In [ ]:
sam.close()
sam.shutdown()

## 关键要点

1. SAM 3无需特定任务训练数据即可对地理空间图像和视频进行零样本分割。

2. 文本、点和框提示满足不同需求，从易用的自然语言查询到精确的空间控制。

3. 置信度分数允许您在下游处理中过滤低质量预测。

4. 批量和切片分割将SAM 3扩展到业务规模和超出GPU内存的大图像。

5. `show_map()`交互式界面无需编写代码即可进行快速探索性分析。

6. `SamGeo3Video`使用流内存架构跨帧连贯地跟踪对象。

7. 整套数据处理流程全程保留地理配准信息，因此输出成果可直接导入 GIS 软件使用。